In [1]:
!pip -q install pinecone sentence-transformers langchain langchain-google-genai pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.4 MB/s eta 0:00:00


In [2]:
import os
import time
import yaml
import numpy as np

from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [3]:
from google.colab import userdata
openai_key = userdata.get('OPENAI_API_KEY')
google_key = userdata.get('GOOGLE_API_KEY')
pinecone_api_key = userdata.get('PINECONE_API_KEY')
# os.environ["GOOGLE_API_KEY"] = google_key

## Steps of Retrieval

### Step 1 - Get your LLM ready

---



---



In [4]:
llm = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash', api_key=google_key)
print(llm.__class__.__name__)

ChatGoogleGenerativeAI


### Step 2 - Access your Pinecone Index and Embedding model

In [5]:
pc = Pinecone(api_key=pinecone_api_key)

index_name = "vector-docs"
index = pc.Index(index_name)

model = SentenceTransformer("paraphrase-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Step 3 - Editing the query (customize)
 - Multi Query Expansion
 - HyDE

In [6]:
query_text = "How do I get a two wheeler loan?"
def edit_query(query: str) -> str:
    # keep simple for now; return as-is
    return query

### Step 4 - Retrieval

In [7]:
# 1. Define the user's question
def retrieve_docs(query_text, top_k = 5):
  query_vector = model.encode([query_text])[0].tolist()
  results = index.query(
      vector=query_vector,
      top_k=5,
      include_metadata=True
  )
  response = []
  # You may tweak here to get whatver you need
  for match in results["matches"]:
      response.append({
          "id": match["id"],
          "score": float(match["score"]),
          "text": match["metadata"]["text"]
      })
  return response

In [10]:
#response

### Step 5 - Context building (customizations)
- Concatenate the context together
- Post filtering
- Add links
- shorten the context
- Summarize


In [11]:
def build_context(results, max_chars: int = 1200) -> str:
    parts = []
    total_chars = 0

    for item in results:
        chunk = item["text"].strip()

        remaining = max_chars - total_chars
        if remaining <= 0:
            break

        if len(chunk) > remaining:
            chunk = chunk[:remaining]

        parts.append(chunk)
        total_chars += len(chunk)

    return "\n\n".join(parts)

In [13]:
# sources = build_context(response, max_chars = 1000)
# sources

### Step 6 - Prompt Rendering and Augmentation (customizations)

In [14]:
PROMPT_TEMPLATE = """
Use the following context to answer the question.

Context:
{sources}

Question:
{query}
""".strip()


def render_prompt(query: str, sources: str) -> str:
    return PROMPT_TEMPLATE.format(query=query_text, sources=sources)

In [16]:
#prompt = render_prompt(query_text, sources)

### Step 7 - Making the Chain

In [17]:
def _start(query):
  return {"query":query}

chain = (RunnableLambda(_start) # {"query": "How do I get a two wheeler loan?"}
         .assign(edited_query = RunnableLambda(lambda d:edit_query( d['query']))) # {'edited_query':'How do I get a two wheeler loan'}
         .assign(results =  RunnableLambda(lambda d:retrieve_docs( d['edited_query'], top_k = 5))) # {'results': All the text from vector DB}
         .assign(sources = RunnableLambda(lambda d: build_context(d['results'], max_chars = 1200))) # {'sources':All sources combined}
         .assign(prompt = RunnableLambda(lambda d: render_prompt(d['edited_query'], d['sources']))) # {'prompt': 'updated prompt'}
         .assign(answer = RunnableLambda(lambda d: llm.invoke(d['prompt'])))
         .pick('prompt') | StrOutputParser()
)

In [19]:
#llm.invoke("What is gen ai?")

In [20]:
query = "What loans can I take?"
chain.invoke(query)

'Use the following context to answer the question.\n\nContext:\nns.\n\n5) Product-Specific FAQs\n\nPersonal Loan\nQ18. What can I use a personal loan for?\nA. Common uses include medical expenses, education, travel, home improvement, wedding expenses, debt consolidation, and urgent cash needs. We do not fund prohibited or illegal purposes.\n\nHome Loan\nQ19. Do you finance new construction, resale, and balance transfer?\nA. Many home loan programs cover purchase of ready property, resale properties, construction, and in certain cases balance transfer from another lend\n\nBrightBridge Finance — FAQs\nSimple loans. Clear terms. Fast decisions.\n\nDisclaimer (read this once):\nThe information below is for general guidance and marketing communication. Final eligibility, interest rates/APR, fees, and terms depend on your profile, internal policy, and applicable laws/regulations. Always review your Key Fact Statement (KFS) / sanction letter and loan agreement before proceeding.\n\n\n1) Getti